# IPL CRUNCH '26 — DATA CLEANING & ANALYSIS
# Competition:
# IPL Crunch '26 by Wooble
#
# Author:
# Yobin Bangera
#

# SECTION 1 — IMPORTING REQUIRED LIBRARIES



In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# SECTION 2 — LOADING THE IPL DATASET

In [2]:
df = pd.read_csv('/content/ipl_matches.csv', low_memory=False)

# SECTION 3 — INITIAL DATA UNDERSTANDING

In [3]:
print("FIRST 5 ROWS OF DATASET")
print(df.head())

print("\nDATASET SHAPE")
print(df.shape)

print("\nDATASET INFORMATION")
print(df.info())

print("\nSUMMARY STATISTICS")
print(df.describe())

print("\nMISSING VALUES")
print(df.isnull().sum())

print("\nDUPLICATE ROWS")
print(df.duplicated().sum())

FIRST 5 ROWS OF DATASET
   match_id        date season                  event  \
0   1082591  2017-04-05   2017  Indian Premier League   
1   1082591  2017-04-05   2017  Indian Premier League   
2   1082591  2017-04-05   2017  Indian Premier League   
3   1082591  2017-04-05   2017  Indian Premier League   
4   1082591  2017-04-05   2017  Indian Premier League   

                                       venue       city                team1  \
0  Rajiv Gandhi International Stadium, Uppal  Hyderabad  Sunrisers Hyderabad   
1  Rajiv Gandhi International Stadium, Uppal  Hyderabad  Sunrisers Hyderabad   
2  Rajiv Gandhi International Stadium, Uppal  Hyderabad  Sunrisers Hyderabad   
3  Rajiv Gandhi International Stadium, Uppal  Hyderabad  Sunrisers Hyderabad   
4  Rajiv Gandhi International Stadium, Uppal  Hyderabad  Sunrisers Hyderabad   

                         team2                  toss_winner toss_decision  \
0  Royal Challengers Bangalore  Royal Challengers Bangalore         field  

# SECTION 4 — DATA CLEANING & PREPROCESSING

In [4]:
# FILTER SEASONS (2022 - 2026)
df['season'] = df['season'].astype(str).str.extract(r'(\d{4})')[0].astype(int)
recent_seasons = [2022, 2023, 2024, 2025, 2026]
df = df[df['season'].isin(recent_seasons)]

In [5]:
# CRITICAL DATA FIX: UNIFY RCB NAMING
print("Fixing franchise naming inconsistencies...")
rcb_old = 'Royal Challengers Bangalore'
rcb_new = 'Royal Challengers Bengaluru'
columns_to_fix = ['batting_team', 'team1', 'team2', 'winner', 'toss_winner']
for col in columns_to_fix:
    if col in df.columns:
        df[col] = df[col].replace(rcb_old, rcb_new)

Fixing franchise naming inconsistencies...


In [6]:
# STANDARDIZE DATATYPES & FILL MISSING VALUES
print("Standardizing datatypes and filling missing values...")
numeric_cols = ['over', 'ball', 'runs_total', 'runs_batter', 'innings', 'runs_extras']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

if 'wicket_kind' in df.columns: df['wicket_kind'] = df['wicket_kind'].fillna('No Wicket')
if 'wicket_player_out' in df.columns: df['wicket_player_out'] = df['wicket_player_out'].fillna('None')
if 'city' in df.columns: df['city'] = df['city'].fillna('Unknown')
if 'player_of_match' in df.columns: df['player_of_match'] = df['player_of_match'].fillna('No Award')

numeric_fill_cols = ['runs_batter', 'runs_total', 'runs_extras', 'extras_wides',
                     'extras_noballs', 'extras_byes', 'extras_legbyes', 'win_by_runs', 'win_by_wickets']
for col in numeric_fill_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

Standardizing datatypes and filling missing values...


In [7]:
# REMOVE DUPLICATES & SUPER OVERS
print("Removing duplicates and Super Overs...")
df.drop_duplicates(inplace=True)
df = df[df['innings'] <= 2]

Removing duplicates and Super Overs...


In [8]:
# FEATURE ENGINEERING
print("Engineering Match Phase and Event features...")

def get_phase(over):
    if over < 6: return 'Powerplay'
    elif over < 15: return 'Middle Overs'
    else: return 'Death Overs'

df['Phase'] = df['over'].apply(get_phase)
df['ball_id'] = df['over'].astype(str) + '.' + df['ball'].astype(str)
df['is_wicket'] = np.where(df['wicket_kind'] != 'No Wicket', 1, 0)
df['is_boundary'] = np.where(df['runs_batter'].isin([4, 6]), 1, 0)
df['boundary_type'] = np.where(df['runs_batter'] == 4, 'Four', np.where(df['runs_batter'] == 6, 'Six', 'No Boundary'))
df['is_dot_ball'] = np.where(df['runs_total'] == 0, 1, 0)

df['toss_match_win'] = np.where(df['toss_winner'] == df['winner'], 1, 0)
df['match_result_type'] = np.where(df['win_by_wickets'] > 0, 'Chased', 'Defended')
df['is_winning_team'] = np.where(df['batting_team'] == df['winner'], 'Won', 'Lost')

# Restoring the 4 missing helper columns from your original notebook
df['run_category'] = np.where(df['runs_batter'] == 0, 'Dot Ball', np.where(df['runs_batter'].isin([1, 2, 3]), 'Running Runs', np.where(df['runs_batter'].isin([4, 6]), 'Boundary', 'Extras')))
df['powerplay_boundary'] = np.where((df['Phase'] == 'Powerplay') & (df['runs_batter'].isin([4, 6])), 1, 0)
df['death_over_runs'] = np.where(df['Phase'] == 'Death Overs', df['runs_total'], 0)

valid_bowler_wickets = ['bowled', 'caught', 'lbw', 'stumped', 'caught and bowled', 'hit wicket']
df['bowler_wicket'] = np.where(df['wicket_kind'].isin(valid_bowler_wickets), 1, 0)

Engineering Match Phase and Event features...


In [9]:
# EXPORT THE SINGLE MASTER DATASET
output_file = 'IPL_Cleaned_Data.csv'
df.to_csv(output_file, index=False)

print("\n================================================")
print(f"MASTER DATASET SAVED AS: {output_file}")
print("================================================")


MASTER DATASET SAVED AS: IPL_Cleaned_Data.csv
